# GPT-2 Fine-Tuning — Emily Prime Corpus

Runs on Google Colab (T4 GPU). Fine-tunes GPT-2 small (124M) on the Emily Prime
training corpus built by `scripts/prime_directive_dataset.py`.

**Open this directly from GitHub — no upload needed:** Colab → File → Open notebook →
GitHub tab → `emilyspringerton/gpt2-alpine-c` → `notebooks/gpt2_finetune_colab.ipynb`.

**Why Colab, not local:** confirmed 2026-07-17 — the dev VM's FatBaby pipeline processes consume
~3GB RSS on a 3.8GB box, leaving no headroom for even the memory-conscious local LoRA trainer
(`scripts/train_local.py`); two attempts were silently OOM-killed within seconds of starting.
Full runbook with corpus-upload options: `docs/COLAB_RUNBOOK.md`.

**Workflow:**
1. Mount Google Drive
2. Load training JSONL from Drive (uploaded by `drive_sync.py`, or manually — see
   `docs/COLAB_RUNBOOK.md` §1 if `GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON` isn't configured)
3. Fine-tune with HuggingFace Trainer
4. Save checkpoint back to Drive
5. Download locally and run `scripts/convert_ft_checkpoint.py`

**Expected training time (T4):** ~20-40 min for 1000 steps on ~5MB corpus. Current corpus
(2026-07-17 build) is 1048 records / 1.3MB — well inside that.

**Target:** entropy delta ≥ 0.5 nats over base GPT-2 (base H_mean=4.4877 nats). A 300-step
local CPU run (2026-06-23) only reached +0.17 nats — this full Colab T4 run is what's needed to
close the gap (NORTHSTAR.md Milestone 3).

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Configure paths — adjust DRIVE_FOLDER to match your Drive folder
DRIVE_FOLDER = '/content/drive/MyDrive/emily-training'  # adjust if different
CORPUS_FILE = os.path.join(DRIVE_FOLDER, 'emily-corpus.jsonl')
OUTPUT_DIR = os.path.join(DRIVE_FOLDER, 'checkpoint-final')
MODEL_NAME = 'gpt2'  # gpt2-small (124M)
MAX_LENGTH = 512
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4  # effective batch = 16
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
WARMUP_STEPS = 100
SAVE_STEPS = 250

print(f'Corpus: {CORPUS_FILE}')
print(f'Output: {OUTPUT_DIR}')

if not os.path.exists(CORPUS_FILE):
    raise FileNotFoundError(
        f'Corpus not found: {CORPUS_FILE}\n'
        'Run scripts/prime_directive_dataset.py locally then upload to Drive with drive_sync.py'
    )

In [ ]:
import json

# Load and inspect corpus
records = []
with open(CORPUS_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} records')
print(f'Sample keys: {list(records[0].keys())}')
print(f'Sample text (first 200 chars): {list(records[0].values())[0][:200]}')

# Detect format
has_text = 'text' in records[0]
has_instruct = 'prompt' in records[0] and 'completion' in records[0]
print(f'Format: {"language-modeling" if has_text else "instruction" if has_instruct else "unknown"}')

In [ ]:
from datasets import Dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Build text column: LM mode or instruct mode
def record_to_text(rec):
    if 'text' in rec:
        return rec['text']
    elif 'prompt' in rec and 'completion' in rec:
        return f"{rec['prompt']}\n\n### Response:\n{rec['completion']}"
    return ''

texts = [record_to_text(r) for r in records if record_to_text(r).strip()]
dataset = Dataset.from_dict({'text': texts})
print(f'Dataset size: {len(dataset)} examples')

def tokenize(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    out['labels'] = out['input_ids'].copy()
    return out

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print(f'Tokenized: {len(tokenized)} examples, max_length={MAX_LENGTH}')

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# 90/10 train/eval split
split = tokenized.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds = split['test']
print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_dir='/content/logs',
    logging_steps=50,
    evaluation_strategy='steps',
    eval_steps=SAVE_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

print('Starting fine-tuning...')
trainer.train()
print('Training complete.')

In [ ]:
# Save final model to Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}')

# Also tar it for easy download
import tarfile, os
tar_path = OUTPUT_DIR + '.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tf:
    tf.add(OUTPUT_DIR, arcname='checkpoint-final')
size_mb = os.path.getsize(tar_path) / 1024 / 1024
print(f'Archived: {tar_path} ({size_mb:.1f} MB)')
print(f'\nDownload manually from Google Drive or use drive_sync.py --download')

In [ ]:
# Quick evaluation: perplexity on eval set
import math
import torch

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f"Eval loss:   {eval_results['eval_loss']:.4f}")
print(f"Perplexity:  {perplexity:.2f}")
print()
print('GPT-2 small base perplexity on WebText: ~29')
print('Lower = model adapted to Emily domain vocabulary')

In [ ]:
# Generation test — verify model produces Emily-domain text
from transformers import pipeline

gen = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0)

prompts = [
    'Emily Prime is the chief of staff for EINHORN_INDUSTRIAL.',
    'The RSI loop begins when obs-watcher detects',
    'An Apple is filed after every backlog completion:',
]

for prompt in prompts:
    out = gen(prompt, max_new_tokens=60, do_sample=True, temperature=0.8, top_p=0.95)
    print(f'PROMPT: {prompt}')
    print(f'OUTPUT: {out[0]["generated_text"]}')
    print()

## Next Steps

After training:
1. Download `checkpoint-final.tar.gz` from Drive
2. Convert to C binary:
   ```bash
   python3 scripts/convert_ft_checkpoint.py \
     --checkpoint checkpoint-final.tar.gz \
     --output weights/emily-ft.bin
   ```
3. Test entropy:
   ```bash
   ./gpt2_run weights/emily-ft.bin --entropy-stats
   ```
4. File a completion Apple:
   ```bash
   emily apples post -t completion "GPT-2 Emily fine-tune complete" "..."
   ```
5. Mark S26-02 done in EMILY/BACKLOG.md